# Create Training Labels from the Critical Mineral Deposits Database
from shapefile to GeoTIFF


In [1]:
import os
import numpy as np
import geopandas as gpd

import sys
if sys.version_info < (3, 9):
    from importlib_resources import files
else:
    from importlib.resources import files

from beak.utilities.io import load_raster
from beak.utilities.conversions import create_binary_raster


# Load data

**User definitions**

In [12]:
# Set base paths and files
BASE_PATH = files("beak.data")

EPSG_CODE = 102008
RESOLUTION = 2240
BASE_SPATIAL = str(EPSG_CODE) + "_" + str(RESOLUTION)
BASE_EXTENT = "us_cont"
BASE_RASTER = BASE_PATH / "BASE_RASTERS" / str("EPSG_" + str(EPSG_CODE) + "_RES_" + str(RESOLUTION) + "_" + BASE_EXTENT + ".tif")

# Points file and query to select relevant mineral occurences
PATH_SHAPEFILE = BASE_PATH / "RAW" / "mineral_deposits" / "Critical_Mineral_Deposits" / "critical_mineral_deposits.shp"
SQL_QUERY = "DepType == 'S-R-V tungsten'"

# Set the output file
PATH_ROOT = BASE_PATH / "PROCESSED" / str("national" + "_" + BASE_SPATIAL + "_" + BASE_EXTENT)
PATH_EXPORT = PATH_ROOT / "labels" / str(BASE_RASTER.stem + "_TUSK_HM6_CMD" + ".tif")
OUT_FILE = PATH_EXPORT

print(f"Output file: {OUT_FILE}")


Output file: s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\PROCESSED\national_102008_2240_us_cont\labels\EPSG_102008_RES_2240_us_cont_TUSK_CMD_SRV_W.tif


# Create Labels


Reproject to **BASE RASTER** CRS

If the reprojection does not work (geometry points are **inf**), run the cell a second time.

In [13]:
from beak.utilities.vector_processing import _reproject_vector_data

base_raster = load_raster(BASE_RASTER)
template_crs = base_raster.crs
gdf = _reproject_vector_data(PATH_SHAPEFILE, crs=template_crs, encoding="utf-8", query=SQL_QUERY)
gdf.head()

,Deposit,Lat_WGS84,Long_WGS84,State,MinSystem,DepType,CritMin,FocusArea,DepCat,Rank,Prduction1,Prduction2,Resources1,Resources2,Resources3,Source_s,Links,geometry
7,Alpha mine,32.36568,-108.37784,NM,Porphyry Cu-Mo-Au,S-R-V tungsten,tungsten,Bound Ranch Mining District,CM-past producer,1,"District production of 4,150 lbs of 1% WO3.",None,Unknown.,None,None,MRDS (2005); McLemore and others (1996),https://mrdata.usgs.gov/mrds/show-mrds.php?dep...,POINT (-1108600.697 -822316.978)
10,American fluorspar deposits,32.38956,-108.37979,NM,Porphyry Cu-Mo-Au,S-R-V tungsten,fluorspar,Bound Ranch Mining District,CM-past producer,1,Estimated production 90.7 metric tons fluorspa...,None,Unknown.,None,None,MRDS (2005); McLemore and others (1996),https://mrdata.usgs.gov/mrds/show-mrds.php?dep...,POINT (-1108413.001 -819542.864)
12,Andrew Curtis mine,34.25692,-117.68380,CA,Porphyry Cu-Mo-Au,S-R-V tungsten,tungsten,Mount Baldy mining district,CM-past producer+resources,5,"Production (1950-1982): 1,100 kg @38.29% WO3.",None,"Inferred resource (1982): 475,000,000 lbs W; I...",None,None,Karl and others (2020),https://doi.org/10.5066/P97NJLI4,POINT (-1880708.188 -458443.088)
18,April Fool-Good Friday,40.00689,-105.40219,CO,Porphyry Cu-Mo-Au,S-R-V tungsten,tungsten,Boulder County tungsten district,CM-past producer,1,"Production (1917-1946): 138,000 stu WO3.",None,Unknown.,None,None,Karl and others (2020),https://doi.org/10.5066/P97NJLI4,POINT (-754423.914 38255.545)
61,Black Pearl,34.68754,-113.03548,AZ,Porphyry Cu-Mo-Au,S-R-V tungsten,tungsten,Basin and Range Proterozoic W veins and replac...,CM-past producer,1,"Production (1951-1956): 62,100 lbs concentrate...",None,Unknown.,None,None,MRDS (2005); Dale (1961),https://mrdata.usgs.gov/mrds/show-mrds.php?dep...,POINT (-1473505.645 -491413.984)


Create binary label raster

In [14]:
out_array = create_binary_raster(gdf, base_raster, query=SQL_QUERY, all_touched=False, same_shape=True, fill_negatives=True, out_file=OUT_FILE)
print(f"Number of positive training labels: {np.sum(out_array==1)}")

Number of positive training labels: 56
